# 02. 크롤링 파이프라인 — Function App · law.go.kr 4종

이 노트북에서는 법령 데이터를 수집합니다.

| # | 크롤러 | 대상 | 사이트 |
|---|--------|------|--------|
| 2 | Function App | 법령/법규명령 (스케줄) | law.go.kr |
| 5-1 | `LawPrecedentCrawler` | 판례 | law.go.kr |
| 5-2 | `HunjaeCrawler` | 헌재결정례 | law.go.kr |
| 5-3 | `ExpCrawler` | 법제처해석례 | law.go.kr |
| 5-4 | `AdmRulCrawler` | 행정심판재결례 | law.go.kr |

law.go.kr 크롤러 4종은 **웹 스크래핑** 방식 (DRF Open API/IP 등록 불필요):
```
목록: POST https://www.law.go.kr/{type}ScListR.do  → HTML 파싱
상세: GET  https://www.law.go.kr/{type}InfoP.do?{type}Seq={id} → HTML 파싱
```

## 1. 환경 설정

In [ ]:
import os
import sys
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

CRAWL_URL    = os.environ.get('AZURE_FUNCTION_CRAWL_URL', '')
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
CONTAINER    = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
RG_NAME      = os.environ.get('AZURE_RESOURCE_GROUP', 'rg-rag-indexing-lab-swc')
LOGIC_APP    = os.environ.get('AZURE_LOGIC_APP_NAME', '')

print(f"Function URL : {CRAWL_URL}")
print(f"Storage      : {STORAGE_NAME}")
print(f"Logic App    : {LOGIC_APP}")

## 2. Function App 수동 트리거 (법령 크롤러)

Logic App 스케줄(21:00 UTC) 대신 즉시 실행이 필요할 때 사용합니다.

In [8]:
if not CRAWL_URL:
    print("AZURE_FUNCTION_CRAWL_URL 미설정 — .env 파일을 확인하세요")
else:
    payload = {"limit": 10, "triggered_by": "notebook-manual"}
    print("크롤링 작업 시작 요청 전송...")
    try:
        # 오래 걸리는 크롤링 완료를 기다리지 않고, 시작/즉시 실패 여부만 확인
        resp = requests.post(CRAWL_URL, json=payload, timeout=(5, 20))
        print(f"HTTP Status: {resp.status_code}")

        is_failed = resp.status_code >= 400
        body = {}
        try:
            body = resp.json() if resp.content else {}
            if isinstance(body, dict):
                if body.get("success") is False:
                    is_failed = True
                if body.get("failed") in (True, 1):
                    is_failed = True
                if isinstance(body.get("failed_count"), int) and body["failed_count"] > 0:
                    is_failed = True
                if body.get("error") or body.get("exception") or body.get("traceback"):
                    is_failed = True
        except ValueError:
            body = {}

        if is_failed:
            print("❌ 시작 단계에서 실패 신호가 감지되었습니다.")
            if body:
                print(json.dumps(body, ensure_ascii=False, indent=2))
        else:
            print("✅ 크롤링 시작 요청은 정상 접수되었습니다.")
            if body:
                print("응답 요약:")
                print(json.dumps(body, ensure_ascii=False, indent=2))
            print("(완료 대기는 하지 않습니다)")

    except requests.exceptions.ReadTimeout:
        # 타임아웃이어도 백엔드가 이미 작업을 시작했을 수 있음
        print("⏱️ 응답 대기 시간이 초과되었습니다.")
        print("✅ 요청은 전달되었을 가능성이 높으며, 백그라운드에서 크롤링이 진행 중일 수 있습니다.")
    except requests.RequestException as e:
        print("❌ 요청 전송 실패 (네트워크/엔드포인트 오류 가능):")
        print(str(e))

⏱️ 응답 대기 시간이 초과되었습니다.
✅ 요청은 전달되었을 가능성이 높으며, 백그라운드에서 크롤링이 진행 중일 수 있습니다.


In [9]:
# 최근 Logic App 실행 중 실패 건수 확인 (크롤링 파이프라인 실패 감시용)
import subprocess
import json as _json

workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    # .env에 이름이 없으면 RG에서 자동 탐색 (1개일 때만 자동 선택)
    discover_cmd = [
        "az", "logic", "workflow", "list",
        "--resource-group", RG_NAME,
        "--query", "[].name",
        "-o", "tsv",
]
    discover_res = subprocess.run(discover_cmd, capture_output=True, text=True)
    if discover_res.returncode == 0:
        names = [n.strip() for n in discover_res.stdout.splitlines() if n.strip()]
        if len(names) == 1:
            workflow_name = names[0]
            print(f"LOGIC_APP 미설정 → 자동 선택: {workflow_name}")
        elif len(names) > 1:
            print("AZURE_LOGIC_APP_NAME 미설정이며 워크플로우가 여러 개입니다. .env에 이름을 지정하세요.")
        else:
            print("RG 내 Logic App 워크플로우를 찾지 못했습니다.")
    else:
        print("워크플로우 자동 탐색 실패:")
        print(discover_res.stderr.strip() or "unknown error")

if workflow_name:
    # 일부 Azure CLI 버전에는 `az logic workflow run list`가 없어 ARM REST API로 조회
    sub_cmd = ["az", "account", "show", "--query", "id", "-o", "tsv"]
    sub_res = subprocess.run(sub_cmd, capture_output=True, text=True)
    if sub_res.returncode != 0 or not sub_res.stdout.strip():
        print("구독 정보 조회 실패:")
        print(sub_res.stderr.strip() or "unknown error")
    else:
        sub_id = sub_res.stdout.strip()
        url = (
            f"https://management.azure.com/subscriptions/{sub_id}"
            f"/resourceGroups/{RG_NAME}"
            f"/providers/Microsoft.Logic/workflows/{workflow_name}/runs"
            f"?api-version=2019-05-01"
        )
        run_cmd = [
            "az", "rest",
            "--method", "get",
            "--url", url,
            "--query", "{total:length(value), failed:length(value[?properties.status=='Failed']), latest:value[0].properties.status}",
            "-o", "json",
]

        run_res = subprocess.run(run_cmd, capture_output=True, text=True)
        if run_res.returncode != 0:
            print("실패 상태 확인 실패 (권한/배포 상태 확인 필요):")
            print(run_res.stderr.strip() or "unknown error")
        else:
            data = _json.loads(run_res.stdout)
            total = data.get("total", 0)
            failed = data.get("failed", 0)
            latest = data.get("latest", "Unknown")

            print(f"최근 실행 건수: {total}")
            print(f"실패 건수    : {failed}")
            print(f"최근 상태    : {latest}")

            if failed > 0:
                print("❌ 최근 실행 중 실패가 있습니다. 실행 이력 상세를 확인하세요.")
            else:
                print("✅ 최근 실행에서 실패가 감지되지 않았습니다.")

LOGIC_APP 미설정 → 자동 선택: logic-crawl-ragi-dyn6dtfu
최근 실행 건수: 5
실패 건수    : 0
최근 상태    : Succeeded
✅ 최근 실행에서 실패가 감지되지 않았습니다.


## 3. Logic App 스케줄 확인

In [10]:
!az logic workflow show \
    --name {LOGIC_APP} \
    --resource-group {RG_NAME} \
    --query "{name:name, state:properties.state}" \
    --output json 2>/dev/null || echo "Logic App 미배포 또는 권한 없음"

Logic App 미배포 또는 권한 없음


# 최근 실행 이력
!az logic workflow run list \
    --name {LOGIC_APP} \
    --resource-group {RG_NAME} \
    --query "[].{status:properties.status, startTime:properties.startTime}" \
    --output table 2>/dev/null || echo "실행 이력 없음"

## 4. Blob Storage 크롤링 결과 확인

In [11]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(CONTAINER)

blobs = list(container_client.list_blobs())

# Blob 경로: {source}/{date}/{source}_{seq}.json
sources = sorted(set(b.name.split('/')[0] for b in blobs if '/' in b.name))

print(f"총 {len(blobs):,}개 파일 | {len(sources)}개 소스 폴더")
for src in sources:
    src_blobs = [b for b in blobs if b.name.startswith(f"{src}/")]
    dates = sorted(set(b.name.split('/')[1] for b in src_blobs if b.name.count('/') >= 2), reverse=True)
    print(f"  /{src}/ : {len(src_blobs)}개 파일, {len(dates)}개 날짜")
    for d in dates[:3]:
        cnt = sum(1 for b in src_blobs if f"{src}/{d}/" in b.name)
        print(f"    {d}: {cnt}개")

ModuleNotFoundError: No module named 'azure.storage'

In [ ]:
import json as _json

# 첫 번째 소스의 최신 JSON 파일 미리보기
if sources:
    src = sources[0]
    json_blobs = sorted(
        [b for b in blobs if b.name.startswith(f"{src}/") and b.name.endswith('.json')],
        key=lambda b: b.name,
        reverse=True,
    )
    if json_blobs:
        blob = container_client.get_blob_client(json_blobs[0].name)
        content = blob.download_blob().readall().decode('utf-8')
        doc = _json.loads(content)
        print(f"파일: {json_blobs[0].name}\n")
        print(_json.dumps(doc, ensure_ascii=False, indent=2)[:3000])
    else:
        print(f"/{src}/ 폴더에 .json 파일 없음")

---
## 5. law.go.kr 크롤러 4종 (로컬 샘플 확인용)

법원 판례, 헌법재판소 결정례, 법제처 법령해석례, 행정심판 재결례를 **로컬에서 샘플 2~3건만 조회**합니다.  
Blob 업로드는 이 섹션에서 수행하지 않습니다.

In [ ]:
from src.crawler.precedent_crawler import (
    LawPrecedentCrawler,
    HunjaeCrawler,
    ExpCrawler,
    AdmRulCrawler,
)
import logging

logging.basicConfig(level=logging.WARNING, format='%(message)s')

### 5-1. 판례 (`LawPrecedentCrawler`)

In [ ]:
prec = LawPrecedentCrawler()

total = prec.get_total_count()
print(f"판례 전체: {total:,}건")

# 목록 1페이지 (100건)
items = list(prec.iter_list(max_pages=1))
print(f"\n[목록 샘플 3건]")
for it in items[:3]:
    print(f"  seq={it['precSeq']:>8} | {it['title_hint'][:70]}")

# 첫 번째 건 상세
doc = prec.get_detail(items[0]['precSeq'])
print(f"\n[상세]")
print(f"  사건명   : {doc['사건명']}")
print(f"  법원명   : {doc['법원명']}")
print(f"  선고일자 : {doc['선고일자']}")
print(f"  사건번호 : {doc['사건번호']}")
print(f"  판시사항 : {doc['판시사항'][:120]}...")
print(f"  URL      : {doc['url']}")

#### 판례 샘플 2~3건 추가 확인 (업로드 없음)

In [ ]:
# 판례 샘플 2~3건만 추가 조회 (Blob 업로드 없음)
sample_docs = []
for it in items[:3]:
    d = prec.get_detail(it['precSeq'])
    if d:
        sample_docs.append(d)
        print(f"  [{d['seq']}] {d['사건명']} ({d['선고일자']})")

print(f"\n샘플 조회 완료: {len(sample_docs)}건 (업로드 없음)")

### 5-2. 헌재결정례 (`HunjaeCrawler`)

In [ ]:
detc = HunjaeCrawler()

total_detc = detc.get_total_count()
print(f"헌재결정례 전체: {total_detc:,}건")

detc_items = list(detc.iter_list(max_pages=1))
print(f"\n[목록 샘플 3건]")
for it in detc_items[:3]:
    print(f"  seq={it['detcSeq']:>8} | {it['title_hint'][:70]}")

# 첫 번째 건 상세
doc_detc = detc.get_detail(detc_items[0]['detcSeq'])
print(f"\n[상세]")
print(f"  사건명   : {doc_detc['사건명']}")
print(f"  재판부   : {doc_detc['재판부']}")
print(f"  사건번호 : {doc_detc['사건번호']}")
print(f"  결정일자 : {doc_detc['결정일자']}")
print(f"  판시사항 : {doc_detc['판시사항'][:120]}...")
print(f"  URL      : {doc_detc['url']}")

# 샘플 2~3건 추가 확인 (업로드 없음)
detc_docs = [detc.get_detail(it['detcSeq']) for it in detc_items[:3]]
detc_docs = [d for d in detc_docs if d]
print(f"\n샘플 조회 완료: {len(detc_docs)}건 (업로드 없음)")

### 5-3. 법제처해석례 (`ExpCrawler`)

In [ ]:
expc = ExpCrawler()

total_expc = expc.get_total_count()
print(f"법제처해석례 전체: {total_expc:,}건")

expc_items = list(expc.iter_list(max_pages=1))
print(f"\n[목록 샘플 3건]")
for it in expc_items[:3]:
    print(f"  seq={it['expcSeq']:>8} | {it['title_hint'][:70]}")

# 첫 번째 건 상세
doc_expc = expc.get_detail(expc_items[0]['expcSeq'])
print(f"\n[상세]")
print(f"  제목     : {doc_expc['제목'][:80]}")
print(f"  문서번호 : {doc_expc['문서번호']}")
print(f"  회시일자 : {doc_expc['회시일자']}")
print(f"  요청기관 : {doc_expc['요청기관']}")
print(f"  질의요지 : {doc_expc['질의요지'][:120]}...")
print(f"  URL      : {doc_expc['url']}")

# 샘플 2~3건 추가 확인 (업로드 없음)
expc_docs = [expc.get_detail(it['expcSeq']) for it in expc_items[:3]]
expc_docs = [d for d in expc_docs if d]
print(f"\n샘플 조회 완료: {len(expc_docs)}건 (업로드 없음)")

### 5-4. 행정심판재결례 (`AdmRulCrawler`)

In [ ]:
admrul = AdmRulCrawler()

total_admrul = admrul.get_total_count()
print(f"행정심판재결례 전체: {total_admrul:,}건")

admrul_items = list(admrul.iter_list(max_pages=1))
print(f"\n[목록 샘플 3건]")
for it in admrul_items[:3]:
    print(f"  seq={it['deccSeq']:>8} | {it['title_hint'][:70]}")

# 첫 번째 건 상세
doc_admrul = admrul.get_detail(admrul_items[0]['deccSeq'])
print(f"\n[상세]")
print(f"  사건명   : {doc_admrul['사건명']}")
print(f"  사건번호 : {doc_admrul['사건번호']}")
print(f"  재결일자 : {doc_admrul['재결일자']}")
print(f"  재결기관 : {doc_admrul['재결기관']}")
print(f"  재결결과 : {doc_admrul['재결결과']}")
print(f"  재결요지 : {doc_admrul['재결요지'][:120]}...")
print(f"  URL      : {doc_admrul['url']}")

# 샘플 2~3건 추가 확인 (업로드 없음)
admrul_docs = [admrul.get_detail(it['deccSeq']) for it in admrul_items[:3]]
admrul_docs = [d for d in admrul_docs if d]
print(f"\n샘플 조회 완료: {len(admrul_docs)}건 (업로드 없음)")

### 5-5. law.go.kr 4종 전체 수집 (CLI)

In [ ]:
# 판례만 수집 (건수 많음 — 수 시간 소요)
# uv run python src/crawler/precedent_crawler.py --source prec --output data/prec.jsonl

# 헌재결정례
# uv run python src/crawler/precedent_crawler.py --source detc --output data/detc.jsonl

# 법제처해석례
# uv run python src/crawler/precedent_crawler.py --source expc --output data/expc.jsonl

# 행정심판재결례
# uv run python src/crawler/precedent_crawler.py --source admrul --output data/admrul.jsonl

# 전체 4종 수집 (--count-only: 건수만 확인)
# uv run python src/crawler/precedent_crawler.py --source all --count-only

print("CLI 명령어 — 터미널에서 직접 실행하세요 (섹션 5에서는 업로드 없음)")

---
## 6. 수집 결과 요약

| 크롤러 | 대상 | 건수 | 수집 필드 |
|--------|------|------|-----------|
| `LawPrecedentCrawler` | 판례 | ~172,000건 | 사건명, 법원명, 선고일자, 판시사항, 판결요지, 전문 |
| `HunjaeCrawler` | 헌재결정례 | ~38,000건 | 사건명, 사건번호, 판시사항, 결정요지, 전문 |
| `ExpCrawler` | 법제처해석례 | ~8,700건 | 제목, 문서번호, 질의요지, 회답, 이유 |
| `AdmRulCrawler` | 행정심판재결례 | ~39,000건 | 사건명, 재결요지, 주문, 이유 |
| Function App | 법령 개정 | 스케줄 수집 | 법령명, 개정일, 내용 |

모든 데이터는 `raw-documents` Blob 컨테이너의 `{source}/{date}/{source}_{seq}.json` 경로에 저장됩니다.

- 날짜 필드는 ISO 8601 형식 (`2024-01-15T00:00:00Z`)으로 저장
- AI Search Indexer가 소스별 prefix (`prec/`, `detc/`, `expc/`, `admrul/`)로 필터링

In [ ]:
# Blob Storage 현황 확인
def count_blobs(container_name: str) -> tuple[int, list[str]]:
    try:
        c = blob_svc.get_container_client(container_name)
        blobs = list(c.list_blobs())
        # 소스(prec/detc/expc/admrul) 단위로 그룹핑
        prefixes = sorted(set(b.name.split('/')[0] for b in blobs if '/' in b.name))
        return len(blobs), prefixes
    except Exception:
        return -1, []

print("Blob Storage 현황:")
for name in ['raw-documents']:
    cnt, prefixes = count_blobs(name)
    status = f"{cnt:,}개" if cnt >= 0 else "컨테이너 없음"
    print(f"  {name}: {status}")
    for p in prefixes:
        print(f"    /{p}/")

---
다음 단계: [03-search-and-query.ipynb](03-search-and-query.ipynb) — AI Search 하이브리드 검색 + GPT-5.4 RAG 질의응답

⚠️ 03 노트북은 **내부망이 연결된 Remote VM 환경에서만 실행**하세요.